# Foundations: Embeddings, Similarity & Classification with Local LLMs

This notebook covers the core building blocks used throughout the rest of this course, all running on **local, open-weight models** via [Ollama](https://ollama.com) instead of a hosted API.

### What you'll build
1. **Text Embeddings** — turn text into a numeric vector with `embeddinggemma`.
2. **Semantic Similarity** — compare two pieces of text by comparing their vectors.
3. **Classification & NER** — use `gemma3:4b` as a lightweight NLP assistant for intent classification and named entity recognition.

---
## Step 1: Install & Start Ollama
Colab doesn't ship with Ollama, so we install it and launch the server as a background process. This takes about a minute.

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import subprocess
import time

# Start the Ollama server in the background
ollama_process = subprocess.Popen(["ollama", "serve"])
time.sleep(5)  # give the server a moment to come up

---
## Step 2: Pull the Models
We need two models:
* `embeddinggemma` — a compact embedding model for Steps 3 & 4.
* `gemma3:4b` — a small chat model for Step 5.

Pulling both can take a few minutes depending on Colab's network speed.

In [ ]:
!ollama pull embeddinggemma
!ollama pull gemma3:4b

In [ ]:
!pip install -q ollama numpy

---
## Step 3: Text Embeddings
An embedding model maps text to a fixed-length vector of numbers that captures its meaning. Semantically similar text ends up with similar vectors.

In [ ]:
import ollama

# Single text embedding
response = ollama.embed(
    model='embeddinggemma',
    input='The quick brown fox jumps over the lazy dog.'
)

# Extract the vector
embedding = response['embeddings'][0]
print(f"Vector Length: {len(embedding)}")
print(f"First 5 values: {embedding[:5]}")

---
## Step 4: Semantic Similarity
Now let's use embeddings to compare two sentences. We embed a *document* and a *query*, then measure how close their vectors are using the dot product.

> **Note:** The dot product works here because both vectors come from the same embedding model. For vectors of differing magnitude, cosine similarity is the more robust choice.

In [ ]:
import numpy as np

def get_vec(text):
    """Fetch embedding vector from local Ollama instance using Gemma."""
    response = ollama.embeddings(model='embeddinggemma', prompt=text)
    return np.array(response['embedding'])

# Generate vectors
doc_vec = get_vec("A fluffy cat is sleeping.")
query_vec = get_vec("A resting feline.")

# Calculate Similarity (Dot Product)
# Note: For truer semantic similarity, consider Cosine Similarity
similarity = np.dot(doc_vec, query_vec)

print(f"Similarity Score: {similarity:.4f}")

---
## Step 5: Classification & Named Entity Recognition (NER)
Beyond embeddings, a small chat model can perform classic NLP tasks directly from a prompt — no training required. Here we use `gemma3:4b` for:
* **Intent Classification** — sorting a user message into a fixed set of categories.
* **Named Entity Recognition** — extracting structured entities (people, orgs, locations, dates) as JSON.

In [ ]:
MODEL = "gemma3:4b"

def run_gemma_task(task_name, prompt, system_prompt="You are a helpful NLP assistant."):
    print(f"\n--- Running Task: {task_name} ---")
    response = ollama.chat(
        model=MODEL,
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': prompt}
        ]
    )
    print(response['message']['content'])

In [ ]:
# Intent Classification
user_msg = "I need to reset my password, I forgot it."
run_gemma_task("Intent Classification", f"Classify this into [Support, Sales, Billing]: {user_msg}")

In [ ]:
# Named Entity Recognition (NER)
ner_text = "Apple CEO Tim Cook visited Mumbai on April 18, 2026."
run_gemma_task("NER", f"Extract (Person, Org, Location, Date) as JSON from: {ner_text}")